In [ ]:
import numpy as np
import torch
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import fbeta_score, average_precision_score
from torch_geometric.datasets import EllipticBitcoinDataset

# 1. Load Data
dataset = EllipticBitcoinDataset(root='../data/Elliptic')
data = dataset[0]

train_idx = data.train_mask.nonzero(as_tuple=True)[0].numpy()
test_idx = data.test_mask.nonzero(as_tuple=True)[0].numpy()

X = data.x.numpy()
# Map target labels from {0, 1} to {-1, +1} for QUBO formulation
y = np.where(data.y.numpy() == 1, 1, -1)

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

# 2. Define the QBoost & QUBO Simulated Annealing Solver Class
class QUBOQBoost:
    def __init__(self, n_estimators=50, max_depth=3, lam=0.05):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.lam = lam
        self.estimators = []
        self.selected_weights = None

    def fit_weak_classifiers(self, X, y):
        """Train a diverse pool of weak decision trees using bootstrap sampling."""
        self.estimators = []
        n_samples = len(X)
        for i in range(self.n_estimators):
            boot_idx = np.random.choice(n_samples, size=n_samples, replace=True)
            tree = DecisionTreeClassifier(max_depth=self.max_depth, max_features="sqrt", random_state=i)
            tree.fit(X[boot_idx], y[boot_idx])
            self.estimators.append(tree)

    def build_qubo_matrix(self, X, y):
        """Construct the Q matrix for binary optimization."""
        M = len(X)
        N = len(self.estimators)
        
        # Matrix H of predictions: shape (M, N)
        H = np.zeros((M, N))
        for j, tree in enumerate(self.estimators):
            H[:, j] = tree.predict(X)

        # Q = (2/M) * H^T * H
        Q = (2.0 / M) * (H.T @ H)
        
        # Adjust diagonal: individual error + sparsity penalty lambda
        diag_err = - (2.0 / M) * (H.T @ y) + self.lam
        np.fill_diagonal(Q, np.diag(Q) + diag_err)
        return Q

    def simulated_annealing_solver(self, Q, steps=10000, temp_start=10.0, temp_min=0.001):
        """Solves w^T Q w using Simulated Annealing on CPU."""
        N = Q.shape[0]
        w = np.random.choice([0, 1], size=N)
        
        def energy(state):
            return state.T @ Q @ state

        current_energy = energy(w)
        best_w = w.copy()
        best_energy = current_energy
        temp = temp_start
        cooling_rate = (temp_min / temp_start) ** (1.0 / steps)

        for step in range(steps):
            flip_idx = np.random.randint(0, N)
            w_next = w.copy()
            w_next[flip_idx] = 1 - w_next[flip_idx]
            
            next_energy = energy(w_next)
            delta = next_energy - current_energy

            # Metropolis acceptance step
            if delta < 0 or np.random.rand() < np.exp(-delta / temp):
                w = w_next
                current_energy = next_energy
                if current_energy < best_energy:
                    best_energy = current_energy
                    best_w = w.copy()

            temp *= cooling_rate

        return best_w

    def fit(self, X, y):
        print(f"1. Training pool of {self.n_estimators} weak decision trees...")
        self.fit_weak_classifiers(X, y)
        
        print("2. Constructing QUBO matrix...")
        Q = self.build_qubo_matrix(X, y)
        
        print("3. Solving QUBO energy minimization via Simulated Annealing...")
        self.selected_weights = self.simulated_annealing_solver(Q)
        
        n_selected = np.sum(self.selected_weights)
        print(f"Optimization Complete: Retained {n_selected}/{self.n_estimators} trees ({n_selected/self.n_estimators:.1%}).")

    def predict(self, X):
        preds = np.zeros(len(X))
        selected_indices = np.where(self.selected_weights == 1)[0]
        
        if len(selected_indices) == 0:
            return np.zeros(len(X))
            
        for idx in selected_indices:
            preds += self.estimators[idx].predict(X)
            
        return np.where(preds >= 0, 1, 0)

# 3. Train & Evaluate QBoost
print("--- Training QBoost Model ---")
qboost = QUBOQBoost(n_estimators=50, max_depth=3, lam=0.05)
qboost.fit(X_train, y_train)

# Evaluate on test set
y_test_binary = np.where(y_test == 1, 1, 0)
qboost_preds = qboost.predict(X_test)

f2_qboost = fbeta_score(y_test_binary, qboost_preds, beta=2)
pr_auc_qboost = average_precision_score(y_test_binary, qboost_preds)

print(f"\n--- QBoost Performance Results ---")
print(f"QBoost F2 Score: {f2_qboost:.4f}")
print(f"QBoost PR-AUC:   {pr_auc_qboost:.4f}")